In [1]:
import pandas as pd
import numpy as np
import xarray as xr
from datetime import datetime
from ICONProcessor import ICONDataGrid

data_dir = '../data_in/'
data_out_dir = '../data_out/'

In [2]:
def get_file(avg=False, sfc=False, pl=False):
    if avg:
        avg_str = '_30min'
    else:
        avg_str = ''
    
    if pl:
        suffix = 'pl__uv'
    elif sfc:
        suffix = 'ml__sfc'
    else:
        suffix = 'ml__lowest5'
    
      
    file = f'{project_root}/{output_dir}/{exp}_LES{avg_str}_51m_{suffix}.nc'
    return file

def format_datetime64(val, format_str='%Y-%m-%d %H:%M:%S'):
    dt = pd.to_datetime(str(val))
    return dt.strftime(format_str)

def get_daterange_str(ds):
    t = pd.to_datetime(ds.time.values,utc=True)
    return f'{t[0].strftime("%Y%m%d")}-{t[-1].strftime("%Y%m%d")}'

## Script settings

In [3]:
scope = 'H3'

project_root = '/work/bb1461/'
exp = 'v370_2030'

In [4]:
if scope == 'H2':
    scope_name = 'HEFEX II'
    output_dir = 'cdo_scripts/output_hefex2/'
    experiment_id =f"hefex2/{exp}/exp_R3B15_51m"
elif scope == 'H3':
    scope_name = 'HEFEX III'
    output_dir = 'cdo_scripts/output_hefex3/'
    experiment_id = f"hefex3/{exp}/exp_R3B15_51m"
else:
    print('Invalid scope!')

loc_file = data_dir + f'{scope}_Coords_w_cells.csv'

## Load and prepare location dimension

In [5]:
df_locs = pd.read_csv(loc_file).sort_values('topo')
df_locs = df_locs[~df_locs['cell_id'].isna()]
df_locs

,cell_id,topo,Original Name,New Name,Type,UAV Takeoff,Lat,Lon,Alt,Comment,Alt_DEM_2022
1,34121,2426.978271,S2,A244,AWS,NaN,46.821410,10.805850,2438.0,from field log,2443.03
3,35219,2523.569824,S1,A245,AWS,NaN,46.816548,10.794520,2451.0,from field log,2456.78
4,35316,2629.500732,Q1,Q255,UAV,NaN,46.811560,10.786100,2550.0,from iMet,2546.79
2,34946,2673.014160,Q2,Q257,UAV,x,46.809810,10.781460,2575.0,from iMet,2583.60
18,37435,2766.071533,Q3,Q267,UAV,x,46.805410,10.774720,2675.0,from iMet,2677.03
17,37400,2766.195557,Q3L,Q267L,UAV,NaN,46.806704,10.772575,2658.0,from iMet,2673.22
5,35693,2798.266357,Q5,Q272L,UAV,NaN,46.802640,10.768980,2702.0,from iMet,2718.42
6,35704,2798.971191,Q4,Q272,UAV,x,46.801750,10.771130,2720.0,from iMet,2726.18
7,35704,2798.971191,Lidar,Lidar,Lidar,NaN,46.801632,10.771678,2720.0,from Streamline GPS,2726.23
8,35704,2798.971191,Tower,T272,AWS,NaN,46.801811,10.771261,2722.0,from SmartFlux GPS,2725.08


In [6]:
dim_loc = df_locs['New Name']
dim_loc = dim_loc.replace('Hintereis','STHE')

dim_loc_lat = np.array(df_locs['Lat'])
dim_loc_lon = df_locs['Lon']
dim_loc_cid = df_locs['cell_id'].astype(int)
dim_loc_ele = df_locs['topo']

## Load and prepare ICON data for locations

In [7]:
def create_data_array(ds, icon_var, height, name, time_dim='time'):
    # get time from dataset
    time = [ICONDataGrid.convert_float_dt(f) for f in ds.time.values]
    time_naive = [dt.replace(tzinfo=None) for dt in time]
    if time_dim.endswith('_avg'):
        t_res = '30min averages (origin is the last value of the timeseries)'
    else:
        t_res = 'Hourly instantaneous values'

    # get data_in for variable (all cell IDs)
    data = ds[icon_var].values[:, height, np.array(dim_loc_cid-1)]

    # create DataArray
    xr_data = xr.DataArray(
        data,
        coords={
            time_dim: (time_dim, time_naive, {"long_name": "Timestamp in UTC"}),
            "location": ("location", dim_loc, {"long_name": "Location name from field campaign"}),
            "lat": ("location", dim_loc_lat, {"long_name": "Latitude", "units": "degrees_north"}),
            "lon": ("location", dim_loc_lon, {"long_name": "Longitude", "units": "degrees_east"}),
            "cid": ("location", dim_loc_cid, {"long_name": "ICON Cell ID of location"}),
            "ele": ("location", dim_loc_ele, {"long_name": "Elevation on ICON grid", "units": "meters"})
        },
        dims=[time_dim, "location"],
        name=name,
        attrs={
            "long_name": ds[icon_var].long_name,
            "standard_name": ds[icon_var].standard_name,
            "units": ds[icon_var].units,
            "temporal_resolution": t_res,
        }
    )
    return xr_data

In [8]:
# surface values
ds = xr.open_dataset(get_file(sfc=True))
xr_t_2m  = create_data_array(ds, 't_2m',  0, 't_2m')
xr_qv_2m = create_data_array(ds, 'qv_2m', 0, 'qv_2m')

# values from lowest model level 
ds = xr.open_dataset(get_file(sfc=False))
xr_t_5m  = create_data_array(ds, 'temp', -1, 't_5m')
xr_qv_5m = create_data_array(ds, 'qv',   -1, 'qv_5m')
xr_u_5m  = create_data_array(ds, 'u',    -1, 'u_5m')
xr_v_5m  = create_data_array(ds, 'v',    -1, 'v_5m')

# values from 500hPa pressure level
ds = xr.open_dataset(get_file(pl=True))
xr_u_500hPa  = create_data_array(ds, 'u', 0, 'u_500hPa')
xr_v_500hPa  = create_data_array(ds, 'v', 0, 'v_500hPa')

# 30min avg values
ds = xr.open_dataset(get_file(avg=True))
xr_u_5m_avg  = create_data_array(ds, 'u', -1, 'u_5m_avg', 'time_avg')
xr_v_5m_avg  = create_data_array(ds, 'v', -1, 'v_5m_avg', 'time_avg')

## Build dataset

In [9]:
# Combine DataArrays
ds_out = xr.Dataset({
    't_2m' : xr_t_2m,
    't_5m' : xr_t_5m,
    'qv_2m': xr_qv_2m,                     
    'qv_5m': xr_qv_5m, 
    'u_5m' : xr_u_5m,
    'v_5m' : xr_v_5m,
    'u_500hPa': xr_u_500hPa,                     
    'v_500hPa': xr_v_500hPa, 
    'u_5m_avg' : xr_u_5m_avg,
    'v_5m_avg' : xr_v_5m_avg,
})

# clean time series (fill gaps and remove duplicates)
ds_out = ds_out.sortby("time")
ds_out = ds_out.sel(time=~ds_out.indexes["time"].duplicated())
time_full = pd.date_range(ds_out.time.min().values, ds_out.time.max().values, freq="1h")
ds_out = ds_out.reindex(time=time_full)

ds_out = ds_out.sortby("time_avg")
ds_out = ds_out.sel(time_avg=~ds_out.indexes["time_avg"].duplicated())
time_full = pd.date_range(ds_out.time_avg.min().values, ds_out.time_avg.max().values, freq="30min")
ds_out = ds_out.reindex(time_avg=time_full)

# Add global attributes
ds_out.attrs = {
    "title": f"Sliced ICON-LES dataset (51m resolution) for {scope_name} locations",
    "institution": "Humboldt-Universität zu Berlin",
    "source": "icon-2024.10 (https://gitlab.dkrz.de/icon/icon-model.git)",
    "history": f"Created {datetime.now().strftime('%Y-%m-%d')}",
    "contact": "alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-6761)",
    "experiment_id": experiment_id,
    "StartTime": format_datetime64(ds_out.time.values[0]),
    "StopTime": format_datetime64(ds_out.time.values[-1]),
    "comment": f"Experiment based on glacier outlines from OGGM SSP370, year 2030"
}

# Write
output_file = data_out_dir + f'HEFEX{scope[1]}_ICON_{exp}_aws_locations_1h_avg30min_{get_daterange_str(ds_out)}.nc'
ds_out.to_netcdf(output_file)
print(f'Output file saved as {output_file}.')

Output file saved as ../data_out/HEFEX3_ICON_v370_2030_aws_locations_1h_avg30min_20250805-20250905.nc.


In [10]:
ds_out

<xarray.Dataset> Size: 788kB
Dimensions:   (time: 763, location: 21, time_avg: 1525)
Coordinates:
  * time      (time) datetime64[ns] 6kB 2025-08-05 ... 2025-09-05T18:00:00
  * location  (location) object 168B 'A244' 'A245' 'Q255' ... 'Q309' 'IHE'
  * time_avg  (time_avg) datetime64[ns] 12kB 2025-08-05 ... 2025-09-05T18:00:00
    lat       (location) float64 168B 46.82 46.82 46.81 ... 46.79 46.79 46.8
    lon       (location) float64 168B 10.81 10.79 10.79 ... 10.74 10.74 10.78
    cid       (location) int64 168B 34121 35219 35316 ... 71796 71796 36558
    ele       (location) float64 168B 2.427e+03 2.524e+03 ... 3.262e+03
Data variables:
    t_2m      (time, location) float32 64kB 278.7 277.1 277.2 ... 270.9 270.1
    t_5m      (time, location) float32 64kB 280.5 279.9 279.5 ... 271.0 270.3
    qv_2m     (time, location) float32 64kB 0.005212 0.004692 ... 0.003649 0.003
    qv_5m     (time, location) float32 64kB 0.005896 0.005696 ... 0.003046
    u_5m      (time, location) float32 64kB 1.887 2.336 2.418 ... 3.63 0.09045
    v_5m      (time, location) float32 64kB -2.285 -3.39 ... -3.617 -2.859
    u_500hPa  (time, location) float32 64kB 2.462 2.476 2.49 ... 13.98 14.48
    v_500hPa  (time, location) float32 64kB -18.89 -18.82 -18.8 ... 1.652 1.23
    u_5m_avg  (time_avg, location) float32 128kB 1.887 2.336 ... 3.637 0.2308
    v_5m_avg  (time_avg, location) float32 128kB -2.285 -3.39 ... -3.571 -2.284
Attributes:
    title:          Sliced ICON-LES dataset (51m resolution) for HEFEX III lo...
    institution:    Humboldt-Universität zu Berlin
    source:         icon-2024.10 (https://gitlab.dkrz.de/icon/icon-model.git)
    history:        Created 2026-04-15
    contact:        alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-946...
    experiment_id:  hefex3/v370_2030/exp_R3B15_51m
    StartTime:      2025-08-05 00:00:00
    StopTime:       2025-09-05 18:00:00
    comment:        Experiment based on glacier outlines from OGGM SSP370, ye...